<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 40px; border-radius: 12px; border: 1px solid #30363d; text-align: center; color: white;">
  <span style="background: rgba(255,255,255,0.2); border: 1px solid rgba(255,255,255,0.4); color: white; padding: 4px 14px; border-radius: 20px; font-size: 12px; font-weight: 600; text-transform: uppercase;">Kafka Training · Lab 8</span>
  <h1 style="color: #ffffff; font-size: 2.4em; font-weight: bold; margin-top: 15px;">Consumer Groups & Scalability</h1>
  <p style="color: #e0e0e0; font-size: 1.1em;">Learn how Kafka distributes load and scales horizontally using a real Python Consumer.</p>
</div>

---

## 🎯 Overview

Kafka is designed for massive scalability. A single application might not be fast enough to process all messages in a high-volume topic. 
By putting multiple consumers in the same **Consumer Group** (`group.id`), Kafka will automatically load-balance the partitions across them!

**Golden Rules of Consumer Groups:**
1. Messages in a single partition are read by only **one** consumer in a group.
2. The maximum number of active consumers in a group equals the **number of partitions**. (Any extra consumers will sit idle as hot standbys).
3. Consumers in **different** groups operate entirely independently (Publish/Subscribe pattern).

---

## ⚙️ Prerequisites

<div style="background-color: rgba(243, 156, 18, 0.1); border-left: 4px solid #f39c12; padding: 10px 15px; margin: 15px 0; border-radius: 4px;">
  <strong>⚠️ Important:</strong> Ensure any previous multi-broker clusters are stopped. Run <code>docker-compose down</code> in Lab7 before proceeding.
</div>

---

## <span style="color: #667eea;">Step 1:</span> Start the Kafka Cluster


In [16]:
!docker-compose -f ../../docker-compose.yml up -d

time="2026-05-20T12:04:07+05:30" level=warning msg="d:\\trainings\\CCDAK-2\\kafka_training\\docker-compose.yml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"
 Container zookeeper  Running
 Container kafka  Running
 Container schema-registry  Running
 Container kafka-connect  Running
 Container control-center  Running


---

## <span style="color: #667eea;">Step 2:</span> Create a Multi-Partition Topic

To distribute load, we MUST have multiple partitions! We will create `group-topic` with **3 partitions**.

In [17]:
!docker exec kafka kafka-topics \
  --bootstrap-server localhost:9092 \
  --create \
  --topic group-topic \
  --partitions 3 \
  --replication-factor 1

Error while executing topic command : Topic 'group-topic' already exists.


[2026-05-20 06:34:17,963] ERROR org.apache.kafka.common.errors.TopicExistsException: Topic 'group-topic' already exists.
 (kafka.admin.TopicCommand$)


---

## <span style="color: #667eea;">Step 3:</span> Review the Python Consumer

Instead of using the console tools, we are going to use the custom `consumer.py` script provided in this folder. 
Open the file and notice how it accepts **Command Line Arguments** for `groupId` and `consumerName`. This allows us to easily spin up multiple instances of the exact same code dynamically!

Let's make sure the required Kafka library is installed in our Jupyter environment.

In [ ]:
%pip install confluent-kafka

---

## <span style="color: #667eea;">Step 4:</span> Start 3 Background Consumers (Group A)

We will use Python's built-in `subprocess` module to start 3 instances of our script concurrently entirely within our Jupyter environment. 
We assign all three to `group-A`, give them unique names (`C1`, `C2`, `C3`), and direct their output into separate local log files (`c1.log`, `c2.log`, `c3.log`).

In [18]:
import subprocess
import time

print("Starting 3 Python consumers in the background...")

p1 = subprocess.Popen(['python', 'consumer.py'], stdout=open('c1.log', 'w'))
p2 = subprocess.Popen(['python', 'consumer.py'], stdout=open('c2.log', 'w'))
p3 = subprocess.Popen(['python', 'consumer.py'], stdout=open('c3.log', 'w'))

consumers = [p1, p2, p3]

# Wait a few seconds for them to connect to the broker and join the group
time.sleep(3)
print("Consumers are running!")

Starting 3 Python consumers in the background...
Consumers are running!


---

## <span style="color: #667eea;">Step 5:</span> Describe the Consumer Group

Let's look at what Kafka did under the hood using the `kafka-consumer-groups` tool.

You should see exactly 3 distinct `CLIENT-ID`s. Kafka has assigned Partition 0 to one consumer, Partition 1 to another, and Partition 2 to the third!

In [25]:
!docker exec kafka kafka-consumer-groups \
  --bootstrap-server localhost:9092 \
  --describe \
  --group group-A


GROUP           TOPIC           PARTITION  CURRENT-OFFSET  LOG-END-OFFSET  LAG             CONSUMER-ID                                  HOST            CLIENT-ID
group-A         group-topic     0          -               0               -               rdkafka-17f4eca6-5dde-4843-b44d-483be400e7d2 /172.28.0.1     rdkafka
group-A         group-topic     1          62              62              0               rdkafka-2ad03f11-f60d-4432-8735-ba48c7505e2c /172.28.0.1     rdkafka
group-A         group-topic     2          538             538             0               rdkafka-2d2d7b41-882e-4bec-bca6-742273a83d52 /172.28.0.1     rdkafka


---

## <span style="color: #667eea;">Step 6:</span> Produce Data and Observe Load Balancing

Now let's produce exactly **300 messages** into the topic.

In [20]:
!docker exec kafka kafka-producer-perf-test \
  --topic group-topic \
  --num-records 300 \
  --record-size 100 \
  --throughput -1 \
  --producer-props bootstrap.servers=localhost:9092

300 records sent, 496.688742 records/sec (0.05 MB/sec), 34.10 ms avg latency, 564.00 ms max latency, 31 ms 50th, 41 ms 95th, 41 ms 99th, 564 ms 99.9th.


Let's count the lines in the local log files! Because Kafka distributes messages across partitions (using a Round-Robin strategy when there are no Keys), our 3 consumers should have shared the workload evenly. 
You should see roughly **100 messages** processed in each log file!

In [ ]:
import time
time.sleep(2) # Give the python scripts a second to flush logs to disk

for f in ['c1.log', 'c2.log', 'c3.log']:
    with open(f, 'r') as file:
        # Count the number of lines (subtracting 1 for the startup 'joined group' message)
        count = sum(1 for line in file) - 1
        print(f"{f} processed {count} messages")

You can even peek inside the log file to see our Python consumer printing out the exact Partition and Offset it processed!

In [23]:
with open('c2.log', 'r') as f:
    for _ in range(5):
        print(f.readline().strip())

Consumer C-20756 joined group: group-A
[C-20756] Processed Order -> Partition: 2 | Offset: 238 | Value: YQHHNEBSEPDNSEIFGAMSUJXKOLTXSPLGHDIOYZJFNIDSPWHZMKVJAXDBZFCOXYKYRJOGYKDESSJMOIIOWVKYUAVWJLXSEPPFEILV
[C-20756] Processed Order -> Partition: 2 | Offset: 239 | Value: BASHCGRHSYGIFSYLVGRXCDVABWWTRQZTMMPBAXGHEPHTASSORYKGVPFGQYJKINSZUJLXQUUDVALUSBFRSXNQHSDFDBAKQZZNTYXF
[C-20756] Processed Order -> Partition: 2 | Offset: 240 | Value: HYGDPYGNRETYAXIXXYQKMKURDSJYIZNEDAHVIVHCJAPGOBQLHUZTKIWTVFEHVYPNGHIDSERMARFXCPYFEPQMFDOTDPWNKMYRMFIA
[C-20756] Processed Order -> Partition: 2 | Offset: 241 | Value: BIQAWWOIFIAKNYFEPTPMIXPQAXFEIKUFFXIDHILBPCBTHWDRMALHFNDCRHAYVLLMRCKJIPNPKGWCIWQCHNHSFSCTYSAKSLVZCCAI


---

## <span style="color: #667eea;">Step 7:</span> The Idle Consumer (Over-provisioning)

What happens if we add a 4th consumer to `group-A`? Because there are only 3 partitions, the 4th consumer will sit idle. It is a "hot standby" and will only take over if one of the first 3 consumers crashes.

Let's start Consumer `C4` and describe the group again. Look closely at the table for the new consumer. It will be listed, but under `ASSIGNMENT`, it won't have any partitions!

In [24]:
print("Starting C4...")
p4 = subprocess.Popen(['python', 'consumer.py', 'group-A', 'C4'], stdout=open('c4.log', 'w'))
consumers.append(p4)

time.sleep(4)
!docker exec kafka kafka-consumer-groups --bootstrap-server localhost:9092 --describe --group group-A

Starting C4...

GROUP           TOPIC           PARTITION  CURRENT-OFFSET  LOG-END-OFFSET  LAG             CONSUMER-ID                                  HOST            CLIENT-ID
group-A         group-topic     0          -               0               -               rdkafka-17f4eca6-5dde-4843-b44d-483be400e7d2 /172.28.0.1     rdkafka
group-A         group-topic     1          62              62              0               rdkafka-2ad03f11-f60d-4432-8735-ba48c7505e2c /172.28.0.1     rdkafka
group-A         group-topic     2          538             538             0               rdkafka-2d2d7b41-882e-4bec-bca6-742273a83d52 /172.28.0.1     rdkafka


---

## 🧹 Clean Up

First, we will politely ask all our background Python scripts to terminate.

In [ ]:
for p in consumers:
    p.terminate()
print("All background consumers successfully terminated.")

Then, stop the cluster and remove volumes to clean up the disk space.

In [ ]:
!docker-compose -f ../../docker-compose.yml down -v

<div style="background-color: rgba(102, 126, 234, 0.1); border: 1px solid rgba(102, 126, 234, 0.3); padding: 20px; text-align: center; border-radius: 8px; margin-top: 40px;">
  <h3 style="color: #667eea; margin-bottom: 10px;">🎉 Lab 8 Complete!</h3>
  <p style="color: #8b949e; margin: 0;">You've successfully mastered Kafka Consumer Groups! You used a real Python application running natively on your machine to scale horizontally, demonstrated load balancing across log files, and proved the rules of partition assignment!</p>
</div>